# 2026 Córdoba Floods — Google Earth Engine Analysis

Companion notebook for the **EcoGeo Tutor** tutorial: *Python: 2026 Córdoba Floods — Google Earth Engine Analysis*.

Reproduces a full flood analysis using Google Earth Engine and free satellite data:
change detection with Sentinel-1 SAR, zonal statistics by municipality, and an
NDVI time series showing agricultural impact.

**Before running:** you'll need a Google Earth Engine account (free) registered at
[code.earthengine.google.com](https://code.earthengine.google.com), and a Google Cloud
project ID to pass to `ee.Initialize()`.


## Setup

Install the Earth Engine and geemap Python clients.

In [ ]:
!pip install earthengine-api geemap -q


## Step 1 — Initialize Earth Engine

Authenticate once per session, then initialize with your own GEE project ID.

In [ ]:
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Authenticate (first time only — opens a browser/consent flow)
ee.Authenticate()

# Initialize with your own GEE project
ee.Initialize(project="your-gee-project-id")

print("GEE initialized successfully")
print("Python EE version:", ee.__version__)


## Step 2 — Define the study area

Load Córdoba department from the FAO GAUL administrative boundary dataset.

In [ ]:
# Load Colombia administrative boundaries (FAO GAUL Level 1)
colombia_depts = ee.FeatureCollection("FAO/GAUL/2015/level1")

# Filter to Córdoba department
cordoba = colombia_depts.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "Colombia"),
        ee.Filter.eq("ADM1_NAME", "Córdoba")
    )
)

cordoba_geom = cordoba.geometry()
print("Córdoba area (km²):", round(cordoba_geom.area().divide(1e6).getInfo(), 1))

# Also define Montería (capital city) for urban focus analysis
municipios = ee.FeatureCollection("FAO/GAUL/2015/level2")
monteria = municipios.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM1_NAME", "Córdoba"),
        ee.Filter.eq("ADM2_NAME", "Montería")
    )
)

# Sinú River basin bounding box (main flood driver)
sinu_basin = ee.Geometry.Rectangle([-76.2, 7.8, -75.0, 9.3])
print("Study area defined")


## Step 3 — Load Sentinel-1 SAR (pre/post flood)

Average multiple scenes to reduce speckle noise.

In [ ]:
# PRE-FLOOD: December 2025 (dry season baseline)
s1_pre = (ee.ImageCollection("COPERNICUS/S1_GRD")
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
            .filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING"))
            .filterBounds(cordoba_geom)
            .filterDate("2025-12-01", "2025-12-31")
            .select("VV")
            .mean()  # average to reduce speckle
            .clip(cordoba_geom))

# POST-FLOOD: February 2026 (peak inundation)
s1_post = (ee.ImageCollection("COPERNICUS/S1_GRD")
             .filter(ee.Filter.eq("instrumentMode", "IW"))
             .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
             .filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING"))
             .filterBounds(cordoba_geom)
             .filterDate("2026-02-01", "2026-02-20")
             .select("VV")
             .mean()
             .clip(cordoba_geom))

# Check statistics
pre_stats  = s1_pre.reduceRegion(ee.Reducer.mean(), cordoba_geom, 30, maxPixels=1e10)
post_stats = s1_post.reduceRegion(ee.Reducer.mean(), cordoba_geom, 30, maxPixels=1e10)
print("Pre-flood mean VV (dB):",  round(pre_stats.getInfo()["VV"], 2))
print("Post-flood mean VV (dB):", round(post_stats.getInfo()["VV"], 2))
# Post should be LOWER — more water = lower backscatter


## Step 4 — Detect flood extent

Change detection thresholded at 3 dB, masked by slope and permanent water, then summarized per municipality.

In [ ]:
# Change detection: pre - post (positive = decrease = new water)
difference = s1_pre.subtract(s1_post)

# Threshold: pixels where backscatter dropped > 3 dB
CHANGE_THRESHOLD = 3.0
flood_raw = difference.gt(CHANGE_THRESHOLD)

# Load DEM for terrain masking
dem = ee.Image("USGS/SRTMGL1_003").clip(cordoba_geom)
slope = ee.Terrain.slope(dem)

# Load JRC permanent water mask (exclude rivers/lakes that are always water)
jrc_water = (ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
               .select("seasonality")
               .clip(cordoba_geom))
permanent_water = jrc_water.gt(10)  # > 10 months/year = permanent

# Final flood mask: changed + flat + NOT permanent water
flood_mask = (flood_raw
              .And(slope.lt(5))           # slope < 5 degrees
              .And(permanent_water.Not()) # exclude permanent water
              .rename("flood"))

# Zonal statistics: flood area per municipality
def get_flood_stats(feature):
    stats = flood_mask.reduceRegion(
        reducer   = ee.Reducer.sum().combine(ee.Reducer.count(), sharedInputs=True),
        geometry  = feature.geometry(),
        scale     = 30,
        maxPixels = 1e9
    )
    flood_px    = ee.Number(stats.get("flood_sum"))
    total_px    = ee.Number(stats.get("flood_count"))
    flood_pct   = flood_px.divide(total_px).multiply(100)
    flood_km2   = flood_px.multiply(30*30).divide(1e6)
    return feature.set({
        "flood_pixels":  flood_px,
        "flood_km2":     flood_km2,
        "flood_pct":     flood_pct,
        "municipio":     feature.get("ADM2_NAME"),
    })

municipios_cordoba = municipios.filter(
    ee.Filter.eq("ADM1_NAME", "Córdoba"))
flood_by_municipio = municipios_cordoba.map(get_flood_stats)

# Export to DataFrame
fc_list   = flood_by_municipio.toList(100)
results   = []
for i in range(fc_list.size().getInfo()):
    feat = ee.Feature(fc_list.get(i))
    props = feat.toDictionary(["municipio","flood_km2","flood_pct"]).getInfo()
    results.append(props)

df = pd.DataFrame(results).sort_values("flood_km2", ascending=False)
print(df.head(10).to_string(index=False))
print(f"\nTotal flood extent: {df['flood_km2'].sum():.0f} km²")


## Step 5 — NDVI time series

Monthly cloud-masked Sentinel-2 composites from January 2025 to June 2026, to quantify vegetation damage and recovery.

In [ ]:
# Cloud-masked Sentinel-2 NDVI function
def mask_s2_clouds(image):
    qa   = image.select("QA60")
    cloud_mask = (qa.bitwiseAnd(1 << 10).eq(0)
                  .And(qa.bitwiseAnd(1 << 11).eq(0)))
    return image.updateMask(cloud_mask).divide(10000)

def get_monthly_ndvi(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end   = start.advance(1, "month")
    s2    = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
               .filterBounds(cordoba_geom)
               .filterDate(start, end)
               .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
               .map(mask_s2_clouds)
               .median()
               .clip(cordoba_geom))
    ndvi = s2.normalizedDifference(["B8", "B4"]).rename("NDVI")
    mean = ndvi.reduceRegion(
        ee.Reducer.mean(), cordoba_geom, 30, maxPixels=1e10
    ).get("NDVI")
    return {"date": f"{year}-{month:02d}", "ndvi_mean": mean.getInfo()}

# Build monthly NDVI from Jan 2025 to Jun 2026
records = []
for year, month in [(2025,m) for m in range(1,13)] + [(2026,m) for m in range(1,7)]:
    try:
        records.append(get_monthly_ndvi(year, month))
    except Exception:
        pass

ndvi_df = pd.DataFrame(records)
ndvi_df["date"] = pd.to_datetime(ndvi_df["date"])
ndvi_df = ndvi_df.dropna()

# Plot with flood event marker
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(ndvi_df.date, ndvi_df.ndvi_mean, "o-",
        color="#22C55E", linewidth=2.5, markersize=7, label="Mean NDVI — Córdoba")
ax.fill_between(ndvi_df.date, ndvi_df.ndvi_mean, alpha=0.12, color="#22C55E")
ax.axvspan(pd.Timestamp("2026-01-31"), pd.Timestamp("2026-02-20"),
           alpha=0.15, color="#EF4444", label="Flood event (Feb 2026)")
ax.axvline(pd.Timestamp("2026-01-31"), color="#EF4444", linestyle="--", linewidth=1.5)
ax.set_ylabel("Mean NDVI", fontsize=12)
ax.set_title("Vegetation Health (NDVI) — Córdoba Department, Colombia\n"
             "Seasonal pattern + flood impact detection", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=30)
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("cordoba_ndvi_timeseries.png", dpi=150)
plt.show()


## Step 6 — Statistics, visualization, and export

Bar chart and pie chart summaries, plus export of the flood raster and per-municipality CSV.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 1. Bar chart: flood area by municipality
top10 = df.head(10)
colors = ["#1D4ED8" if v > 100 else "#93C5FD" for v in top10.flood_km2]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].barh(top10.municipio[::-1], top10.flood_km2[::-1], color=colors[::-1])
axes[0].set_xlabel("Flooded area (km²)", fontsize=12)
axes[0].set_title("Top 10 Most Affected Municipalities\nCórdoba Floods — February 2026",
                  fontsize=13, fontweight="bold")
axes[0].axvline(100, color="#EF4444", linestyle="--", linewidth=1, alpha=0.7)
for i, (v, n) in enumerate(zip(top10.flood_km2[::-1], top10.municipio[::-1])):
    axes[0].text(v + 2, i, f"{v:.0f} km²", va="center", fontsize=10)

# 2. Pie chart: flood severity classes
labels   = ["High (>200 km²)", "Medium (50-200 km²)", "Low (<50 km²)"]
sizes    = [
    (df.flood_km2 > 200).sum(),
    ((df.flood_km2 >= 50) & (df.flood_km2 <= 200)).sum(),
    (df.flood_km2 < 50).sum()
]
colors_p = ["#1D4ED8", "#93C5FD", "#BFDBFE"]
axes[1].pie(sizes, labels=labels, colors=colors_p, autopct="%1.0f%%",
            startangle=90, textprops={"fontsize": 11})
axes[1].set_title("Municipalities by Flood Severity Class\nCórdoba — February 2026",
                  fontsize=13, fontweight="bold")

plt.suptitle("Córdoba Flood Analysis — Google Earth Engine + Sentinel-1",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("cordoba_flood_stats.png", dpi=150, bbox_inches="tight")
plt.show()

# 3. Export results
df.to_csv("cordoba_flood_by_municipio.csv", index=False)
print("Saved: cordoba_flood_by_municipio.csv")
print(f"\n{'='*50}")
print("SUMMARY - Cordoba Floods February 2026")
print(f"{'='*50}")
print(f"Total flooded area:        {df.flood_km2.sum():.0f} km²")
print(f"Most affected municipality: {df.iloc[0].municipio} ({df.iloc[0].flood_km2:.0f} km²)")
print(f"Municipalities affected:    {(df.flood_km2 > 5).sum()} of {len(df)}")

# Export flood raster to Google Drive
task = ee.batch.Export.image.toDrive(
    image       = flood_mask.toFloat(),
    description = "Cordoba_Flood_Feb2026_SAR",
    folder      = "GEE_Exports",
    fileNamePrefix = "cordoba_flood_2026",
    region      = cordoba_geom,
    scale       = 30,
    crs         = "EPSG:4326",
    maxPixels   = 1e10
)
task.start()
print("Export to Google Drive started - check Tasks in GEE Code Editor")


## Validation

Compare your flood map against official reference data:

- **Copernicus EMS EMSR865** — [mapping.emergency.copernicus.eu](https://mapping.emergency.copernicus.eu) — the official rapid mapping product for this exact event.
- **JRC Global Surface Water** — [global-surface-water.appspot.com](https://global-surface-water.appspot.com) — baseline permanent water extent.
- **UNGRD Colombia** — [portal.gestiondelriesgo.gov.co](https://portal.gestiondelriesgo.gov.co) — affected municipality lists for ground-truth comparison.

A good result for this type of SAR flood detection typically achieves overall accuracy above 85% against reference flood perimeters.

---
*Companion notebook for the EcoGeo Tutor tutorial on rcafe.vercel.app*
